# 05: molecular variability, the embedding

### CAJAL NEUROMICS summer school, Bordeaux 2026 · a spatial metabolomics primer

*Luca Fusar Bassini · hands-on notebook 5 of 9 · ~90 minutes*

---

By now you have two coronal sections of mouse brain, the same plane at about anteroposterior level
6.5, one from a control female and one from a pregnant female, each pixel carrying 173 lipid
intensities. That is a table with 174,768 rows and 173 columns. You cannot stare at 173 columns and
see structure. This notebook turns those 173 numbers per pixel into a handful of interpretable
**lipid programs**, and along the way teaches the three ideas the whole atlas pipeline rests on:

> **feature selection by Moran's I → NMF (the embedding) → t-SNE / UMAP (just to look) → Harmony (only for clustering)**

We do every step in the most transparent way first: a few lines of plain code so you see the gears
turn, then the matching one-liner from our course helper `cajal_lipidomics`. The helper is the same
thing, made robust. The plots are the teaching. We look at the data before, during, and after every
transformation, because a number you cannot see is a number you cannot trust.

A reminder of the four callouts that run through every notebook:

- 🔬 **TASK**: something you do (write or run code).
- 💡 **HINT**: a nudge when you are stuck.
- ❓ **QUESTION**: pause and think, no code required.
- ⚠️ **CHECKPOINT**: what you should see if it worked. If your screen disagrees, stop and fix it.


## set up the tools

We pull in the scientific-Python stack, the course helper package `cajal_lipidomics` (imported as
`cl`), and the one global seed that makes every number and figure below reproducible. `set_style()`
applies the lab's clean figure defaults so your plots match the atlas.

🔬 **TASK.** Run the next cell. You should see a short `ready` line and no red error.

In [ ]:
# the stack you already know
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad

# the course helper package: transparent recipes mirroring the EUCLID/atlas pipeline
import cajal_lipidomics as cl
from cajal_lipidomics import analysis, embedding, plotting
from cajal_lipidomics.style import set_style, FS
set_style()

# one seed so every figure and number below is reproducible
SEED = 0
rng = np.random.default_rng(SEED)

print(f"ready: numpy {np.__version__} | pandas {pd.__version__} | anndata {ad.__version__}")

⚠️ **CHECKPOINT.** You see `ready: numpy ... | pandas ... | anndata ...`. A `ModuleNotFoundError`
means the wrong kernel: pick the `cajal-lipidomics` kernel in the top-right and rerun.

## load the two sections

We load the paired-section dataset into an **AnnData** object, the standard container for this kind of
data. Think of it as a smart spreadsheet with three parts that always travel together:

- `adata.X` is the **measurements**: a matrix with one row per pixel and one column per lipid. These
  values are uMAIA-normalized intensities, the output of the normalization you met earlier.
- `adata.var_names` are the **column names**: the 173 lipid names, written in lipid shorthand like
  `PC 38:6` (a phosphatidylcholine with 38 total acyl carbons and 6 double bonds).
- `adata.obs` is the **per-pixel metadata**: which section a pixel came from (`SectionID`), its
  condition (`Condition`: `naive` for control, `pregnant`), its position in the Allen common
  coordinate framework (`zccf`, `yccf`), the Allen brain region it falls in (`acronym`), and the
  precomputed lipizone label (`lipizone_names`) we will use only as a backdrop.

One pixel is one MALDI laser spot, not one cell. It mixes cell bodies, axons, dendrites, and glial
projections in a tiny patch of tissue. Keep that in mind: a pixel is a local average of membranes.

In [ ]:
adata = ad.read_h5ad("../../data/sections_pair.h5ad")

print("pixels x lipids:", adata.shape)
print("conditions:", dict(adata.obs["Condition"].value_counts()))
print("sections:", dict(adata.obs["SectionID"].value_counts()))
print("first lipids:", list(adata.var_names[:6]))

⚠️ **CHECKPOINT.** `pixels x lipids: (174768, 173)`, two conditions (`naive` 84,321 pixels,
`pregnant` 90,447), and the first lipid names look like real lipids. The control section is `217D`
(SectionID 75), the pregnant is `Brain1_C2` (SectionID 110), and they sit at the same coronal plane so
they are anatomically comparable.

Before any analysis, look at the raw data. We plot one lipid across both sections so you see what a
single column of `adata.X` looks like spread over the tissue. `HexCer 42:2;O2` is a
hexosylceramide, a sphingolipid that marks **myelin**, the fatty insulation around axons that fills
white matter. If everything is wired up correctly it should paint the white-matter tracts.

In [ ]:
# spatial map of one myelin lipid across both sections (helper does the publication styling)
plotting.spatial_lipid(adata, "HexCer 42:2;O2", point_size=3.0)
plt.show()

⚠️ **CHECKPOINT.** Two brain-shaped scatters appear, control on the left, pregnant on the right.
The bright (high-intensity) pixels trace the fibre tracts: the corpus callosum arching over the top,
the internal capsule, the fimbria. That bright skeleton is myelin. The fact that one lipid alone draws
recognisable anatomy is the whole promise of spatial lipidomics, and it is also the property we are
about to exploit to separate signal from noise.

---
## 1 · feature selection: keep the lipids that paint anatomy

**The problem.** Of our 173 lipids, some draw crisp anatomy like the myelin map you just saw, and some
are mostly measurement noise: speckle that flickers pixel to pixel with no spatial pattern. Feeding
noise into the embedding wastes factors on junk and blurs the real structure. So before we compress,
we **select features**: keep the lipids that carry real spatial signal, drop the ones that do not.

**The idea.** A real biological signal is **spatially smooth**: if a pixel is high in some lipid, its
neighbours tend to be high too, because tissue is organised into regions, not random dots. Noise has no
such smoothness. So we need one number per lipid that says "do nearby pixels have similar values?".
That number is **Moran's I**, the workhorse measure of spatial autocorrelation.

### what Moran's I actually computes

Let us build it from scratch so it is never a black box. For one lipid we have a value $x_i$ at every
pixel $i$, and the pixel coordinates $(z_i, y_i)$. The recipe has four moves.

**1. Define who is a neighbour.** For each pixel, find its $k$ nearest pixels in space (we use
$k = 6$). That gives a **k-nearest-neighbour graph**: a network where each pixel is wired to its 6
closest neighbours. The weight $w_{ij}$ is 1 if $j$ is a neighbour of $i$, else 0.

**2. Centre the values.** Subtract the mean: $\,\tilde{x}_i = x_i - \bar{x}$. Now positive means
"above average for this lipid", negative means "below".

**3. Multiply each pixel by its neighbours.** For every edge in the graph, form the product
$\tilde{x}_i \,\tilde{x}_j$. If a pixel and its neighbour are both above average (or both below), the
product is **positive**. If one is above and the other below, it is **negative**. Sum over all edges.

**4. Normalise.** Divide by the total variance $\sum_i \tilde{x}_i^2$ and scale by $N/W$ (pixels over
total edge weight). The result is

$$ I \;=\; \frac{N}{W}\,\frac{\sum_{i}\sum_{j} w_{ij}\,(x_i-\bar{x})(x_j-\bar{x})}{\sum_i (x_i-\bar{x})^2}. $$

Read it as a correlation between a pixel and its neighbours. $I \approx 1$: neighbours look alike,
strong spatial pattern. $I \approx 0$: neighbours are unrelated, salt-and-pepper noise. The dot
products in the numerator are exactly the dot products you met in linear algebra, here measuring
agreement between a pixel and the pixels around it.

🔬 **TASK.** Compute Moran's I for one lipid, by hand, in a few transparent lines. We do it on the
control section only (we learn everything on the control brain, then transfer, just as the atlas does).
We use `scipy`'s `cKDTree` to find each pixel's 6 nearest spatial neighbours, then assemble the formula
term by term.

In [ ]:
# 🔬 your code here


💡 **HINT.** `cKDTree(coords).query(coords, k=7)` returns, for each pixel, the indices of its 7
nearest points sorted by distance. The first is the pixel itself (distance 0), so we slice off column 0
with `idx[:, 1:]` to keep the 6 genuine neighbours.

The helper `cl.analysis.morans_i(values, coords, k=6)` does exactly these four steps. Let us confirm it
matches our hand computation, then run it over all 173 lipids at once.

In [ ]:
# the helper reproduces our hand-rolled value
moran_helper = analysis.morans_i(X_ctrl[:, j], coords, k=6)
print(f"hand-rolled: {moran_hex:.4f}   helper: {moran_helper:.4f}")

# score every lipid in the control section
moran_all = np.array([analysis.morans_i(X_ctrl[:, c], coords, k=6) for c in range(X_ctrl.shape[1])])
moran_series = pd.Series(moran_all, index=names).sort_values()

print("\nlowest Moran's I (noisiest lipids):")
print(moran_series.head(4).round(3).to_string())
print("\nhighest Moran's I (cleanest, most anatomical):")
print(moran_series.tail(4).round(3).to_string())

⚠️ **CHECKPOINT.** The helper matches the hand computation to four decimals. The myelin lipid
`HexCer 42:2;O2` sits near the very top (Moran ≈ 0.93), while lipids like `LPC 16:0` sit near the
bottom (Moran ≈ 0.23). The whole distribution skews high, which tells us most of these 173 lipids are
already good: this is a curated, high-confidence panel, not raw peaks.

Now the proof that Moran's I means what we claim. We put the **best** Moran lipid next to the **worst**
as spatial maps. High Moran should look like anatomy, low Moran should look like static.

In [ ]:
# pick the highest- and lowest-Moran lipids and map them side by side, control section only
best = moran_series.index[-1]
worst = moran_series.index[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, lip in zip(axes, [best, worst]):
    v = X_ctrl[:, list(names).index(lip)]
    vmin, vmax = np.quantile(v, 0.02), np.quantile(v, 0.98)
    sc = ax.scatter(ctrl.obs["zccf"], -ctrl.obs["yccf"], c=v, cmap="plasma",
                    s=2, vmin=vmin, vmax=vmax, rasterized=True)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(f"{lip}\nMoran's I = {moran_series[lip]:.2f}", fontsize=FS["m"])
fig.suptitle("high Moran = real anatomy   vs   low Moran = noise", fontsize=FS["l"])
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** Left: the high-Moran lipid draws clean, connected structures, you can see the
brain's anatomy. Right: the low-Moran lipid is a noisy haze with no coherent pattern. Same tissue, same
pixels, wildly different spatial coherence. **That** is what Moran's I quantifies, and why it is the
right filter.

❓ **QUESTION.** The low-Moran lipid is not necessarily fake. It might be a real molecule that simply is
not regionally organised, or one measured close to the detection limit so noise dominates. Either way,
it carries no *spatial* structure for our regional analysis. Why is "no spatial structure" a good
reason to drop a feature before we look for spatial lipid programs?

🔬 **TASK.** Apply the filter. The atlas used a permissive threshold of Moran's I > 0.4 to drop
the noisiest features before embedding. We do the same: keep every lipid above 0.4, and remember which
ones we kept. This is the feature-selected panel that feeds the embedding.

In [ ]:
# keep lipids whose spatial signal clears the threshold
MORAN_THR = 0.4
keep_mask = moran_all > MORAN_THR
kept_lipids = list(names[keep_mask])

print(f"kept {keep_mask.sum()} of {len(names)} lipids at Moran's I > {MORAN_THR}")
print("dropped:", sorted(names[~keep_mask].tolist()))

# the working tables: same pixels, only the selected lipids
ctrl_sel = ctrl[:, kept_lipids].copy()
adata_sel = adata[:, kept_lipids].copy()
print("control selected matrix:", ctrl_sel.shape)

⚠️ **CHECKPOINT.** About 158 lipids survive and roughly 15 are dropped, the noisy ones like
`LPC 16:0` and a couple of triglycerides. In the full atlas this same logic, combined with a dropout
filter and a variance criterion across many sections, trimmed thousands of raw peaks down to 105
clean lipids. With only two sections we lean on Moran's I alone, which is the single most important of
those criteria. The helper `cl.analysis.morans_i` is exactly the per-lipid scorer; the threshold is a
choice you state and defend.

---
## 2 · NMF: compress 158 lipids into a few additive programs

We have a clean panel, but it is still 158 columns wide. We want to compress it into a handful of
**lipid programs**, co-varying groups of lipids that act as a unit, so that downstream we cluster and
visualise a few interpretable numbers per pixel instead of 158 raw intensities. The tool is
**non-negative matrix factorisation**, NMF.

### the linear algebra, from the ground up

Our data is a matrix $V$ with shape (pixels × lipids), every entry an intensity. NMF approximates it as
a product of two **smaller, non-negative** matrices:

$$ V \;\approx\; W \, H. $$

- $H$ has shape (programs × lipids). Each **row** is one program: a recipe of non-negative weights
  saying how much each lipid belongs to that program. One program might be "lots of HexCer and SM",
  the myelin recipe; another "lots of PC and PE", a grey-matter membrane recipe.
- $W$ has shape (pixels × programs). Each **row** is one pixel, and the entries say **how much of each
  program that pixel expresses**, again all non-negative.

To reconstruct one pixel's full lipid profile you take its program activities (a row of $W$) and mix
the recipes (the rows of $H$) in those amounts. In matrix terms, pixel $i$'s reconstructed profile is
$\sum_k W_{ik}\,H_{k\cdot}$, a weighted sum of recipes. That is a **dot product** per lipid: the
row of $W$ dotted with a column of $H$. Linear algebra you already know, doing real biological work.

**Why non-negative matters.** Because nothing is ever subtracted, every pixel is literally a *sum of
parts, each present in some non-negative amount*. You can never have "minus 3 units of the myelin
program". This is what makes the factors readable as biology. The famous demonstration is NMF on face
photographs: it discovers parts like a nose, an eyebrow, a mouth, because faces are built by adding
parts, not subtracting them.

### NMF versus PCA, the cousin you met before

You met PCA earlier. Both compress a wide table into a few new coordinates, but they differ in one
decisive way:

- **PCA** finds orthogonal directions of maximum variance. Its components freely mix lipids with **plus
  and minus signs**, so a component can say "more PC *minus* SM". Mathematically elegant, biologically
  awkward: a negative amount of a lipid has no meaning.
- **NMF** forbids negatives everywhere. Its programs are **purely additive parts**. You lose PCA's
  orthogonality and its variance-ranking, but you gain components you can almost name.

For *interpretation* ("which lipids define this territory?") NMF wins. For raw variance accounting PCA
wins. The atlas uses NMF precisely because the goal is interpretable lipid programs.

We do not want to take that sign difference on faith, because a number you cannot see is a number you
cannot trust. So we will *show* it: in a moment, on the very same toy matrix, we fit both PCA and NMF
and print their components side by side. You will read the minus signs in PCA's components and the
all-positive weights in NMF's recipes, the abstract claim made concrete in two small arrays.

### a tiny worked example so the shapes are concrete

Before touching the real data, factor a tiny 4-pixel × 3-lipid toy matrix into 2 programs, so you can
see $V$, $W$, and $H$ as small printed arrays and check by eye that $W H$ rebuilds $V$.

In [ ]:
from sklearn.decomposition import NMF

# 4 pixels, 3 lipids. Pixels 0-1 are "myelin-like" (high lipid C), pixels 2-3 "grey-like" (high A,B)
V = np.array([[0.1, 0.2, 9.0],
              [0.2, 0.1, 8.0],
              [7.0, 6.0, 0.3],
              [8.0, 7.0, 0.2]])

toy = NMF(n_components=2, init="nndsvda", random_state=SEED, max_iter=500)
W_toy = toy.fit_transform(V)     # (4 pixels x 2 programs)
H_toy = toy.components_          # (2 programs x 3 lipids)

print("V (pixels x lipids):\n", V)
print("\nW (pixels x programs), how much of each program per pixel:\n", W_toy.round(2))
print("\nH (programs x lipids), the two recipes:\n", H_toy.round(2))
print("\nW @ H reconstructs V:\n", (W_toy @ H_toy).round(2))

⚠️ **CHECKPOINT.** `W @ H` lands almost exactly on `V`. Read the rows of `W`: pixels 0 and 1 load
heavily on one program, pixels 2 and 3 on the other, and the two rows of `H` show one recipe dominated
by lipid C and one by lipids A and B. NMF rediscovered the two pixel types and the two lipid recipes
with no labels, just from the requirement that everything be non-negative and add up.

### the same toy matrix through PCA, so the sign difference is shown not asserted

We claimed NMF and PCA differ in their signs. Let us prove it on the exact same toy `V`. We fit
scikit-learn's `PCA(n_components=2)` and print its `components_` next to NMF's recipes `H_toy` from the
cell above. Watch the signs: PCA's loadings carry minus signs (it is allowed to say "this direction is
high in lipid C *minus* lipids A and B"), while every weight in NMF's `H` is zero or positive (it can
only *add* lipids). That is the whole interpretability argument, in two printed arrays.

In [ ]:
from sklearn.decomposition import PCA

# fit PCA on the SAME toy V, ask for 2 components, and compare its loadings to NMF's recipes
pca_toy = PCA(n_components=2, random_state=SEED).fit(V)

print("NMF recipes H (programs x lipids), the additive parts:\n", H_toy.round(2))
print("every NMF weight >= 0 ?", bool((H_toy >= 0).all()))
print("\nPCA components (components x lipids), free to be positive OR negative:\n",
      pca_toy.components_.round(2))
print("any PCA loading < 0 ?", bool((pca_toy.components_ < 0).any()))

### seeding: a smarter start than random

Plain NMF starts from a random or generic guess and can land in a poor local optimum, giving factors
that differ run to run. The atlas uses a **seeded** start: group the lipids by how they correlate,
take the most representative lipid of each group as the seed for one program, and initialise $W$ and
$H$ from those seeds. The intuition: lipids that rise and fall together belong to the same program, so
we hand NMF that structure up front instead of making it rediscover it from noise.

Our helper `cl.embedding.seeded_nmf` implements exactly this:

1. compute the lipid-lipid correlation matrix and turn it into a distance ($1 - |\text{corr}|$, taking
   the absolute value because anticorrelated lipids still carry the same structural information),
2. cluster the lipids into `n_factors` groups by that distance,
3. pick the most central lipid of each group as a seed program,
4. run scikit-learn's `NMF` with `init="custom"` from those seeds.

🔬 **TASK.** Learn the embedding on the **control** section. We ask for 12 programs. The atlas chose
its factor count (16) data-driven from the number of lipid correlation communities; 12 is a sensible
fixed choice for our two sections that keeps every factor easy to inspect.

In [ ]:
# learn the seeded NMF on the CONTROL section's selected lipids
# 🔬 your code here


⚠️ **CHECKPOINT.** `W` is (84321 × 12), one row per control pixel with its 12 program activities;
`H` is (12 × 158), one row per program with its lipid recipe; and every entry is non-negative. We have
turned 158 lipid columns into 12 program columns per pixel.

You may also see a red `ConvergenceWarning: Maximum number of iterations 400 reached`. That is expected
and harmless here. NMF keeps polishing $W$ and $H$ by tiny amounts long after the factorisation has
already settled into its shape, so scikit-learn warns that it stopped at the iteration cap rather than
at a perfectly flat optimum. The programs are already stable: pushing `max_iter` higher shifts the
numbers below by less than a percent and changes none of the conclusions. If the red text bothers you,
pass a larger `max_iter`, but you do not need to.

Now interpret one program in both directions: its **recipe** (a row of $H$, the top lipids) and its
**spatial map** (a column of $W$, painted over the tissue). A program is only useful if those two views
agree, a coherent set of lipids that lights up a coherent piece of anatomy.

In [ ]:
# rank programs by how much spatial signal they carry (variance of their per-pixel activity)
prog_var = W_ctrl.var(axis=0)
prog_order = np.argsort(prog_var)[::-1]
sel_names = np.array(ctrl_sel.var_names)

# show the 3 strongest programs: top lipids (H) on the left, spatial map (W) on the right
fig, axes = plt.subplots(3, 2, figsize=(11, 11),
                         gridspec_kw={"width_ratios": [1.1, 1.0]})
for r, p in enumerate(prog_order[:3]):
    top = np.argsort(H[p])[::-1][:8]                 # 8 highest-weighted lipids of this program
    axes[r, 0].barh(range(8), H[p][top][::-1], color="mediumpurple")
    axes[r, 0].set_yticks(range(8))
    axes[r, 0].set_yticklabels(sel_names[top][::-1], fontsize=FS["xs"])
    axes[r, 0].set_title(f"program {p}: lipid recipe (H)", fontsize=FS["m"])
    axes[r, 0].set_xlabel("weight")

    w = W_ctrl[:, p]
    vmax = np.quantile(w, 0.98)
    sc = axes[r, 1].scatter(ctrl.obs["zccf"], -ctrl.obs["yccf"], c=w, cmap="plasma",
                            s=2, vmin=0, vmax=vmax, rasterized=True)
    axes[r, 1].set_aspect("equal"); axes[r, 1].set_xticks([]); axes[r, 1].set_yticks([])
    for s in axes[r, 1].spines.values():
        s.set_visible(False)
    axes[r, 1].set_title(f"program {p}: activity per pixel (W)", fontsize=FS["m"])
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** For each of the three strongest programs the recipe and the map line up. A
program dominated by particular phosphatidylcholines paints one set of territories; a program built
from other lipids paints a different set. The activity maps are spatially coherent, not speckled,
which is the payoff of having filtered on Moran's I first. Each program is a "molecular signature" you
could hand a biologist and say: these lipids, here.

💡 **HINT.** We sorted programs by `W.var(axis=0)`, the spread of a program's activity across pixels. A
program that is roughly constant everywhere carries little spatial information and ranks low; a program
that is high in some regions and low in others ranks high. This is the same logic as Moran's I and PCA:
structure lives in variance.

### apply the learned embedding to both sections

The atlas learns NMF on one brain and then **applies** it everywhere, so every pixel in every brain is
described in the *same* 12-program vocabulary. That shared vocabulary is what lets us later compare
control and pregnant pixels and transfer labels between them. Applying NMF means: keep $H$ (the
recipes) fixed, and solve for each new pixel's program activities $W$. Scikit-learn does this with
`model.transform`, wrapped by `cl.embedding.apply_nmf`.

🔬 **TASK.** Project all 174,768 pixels (both sections) into the 12-program space the control section
defined, and store the result in `adata_sel.obsm["X_NMF"]`.

In [ ]:
# project BOTH sections into the control-learned program space (H stays fixed)
# 🔬 your code here


⚠️ **CHECKPOINT.** The embedding is (174768 × 12), and on the control rows every factor correlates
above 0.97 with the $W$ we learned: the projection is consistent on the data it was trained on, as it
must be. It is not bit-for-bit identical because applying the model re-centres the inputs using the
full two-section minimum rather than the control-only minimum, a tiny offset that leaves the structure
untouched. Every pixel in both sections now lives in the same 12-dimensional program space. From here
on we work in that space, not in raw lipids.

---
## 3 · t-SNE and UMAP: a flat picture, for looking only

We have 12 program dimensions per pixel. Screens are flat. To *see* the structure we squash those 12
dimensions into 2 with a **neighbourhood-preserving** map: t-SNE or UMAP. Both ask "who are each
pixel's near neighbours in the 12-D program space?" and try to keep those same pixels close together
in 2-D, letting faraway pixels fall where they may. The result is a scatter where tight blobs are
genuinely similar pixels.

**We always run these on the embedding, never on raw lipids.** Three reasons: it is far faster on 12
columns than on 158, the embedding has already denoised the data, and raw high-dimensional distances
are unreliable (in many dimensions everything drifts roughly equidistant, the "curse of
dimensionality"). Compress first with NMF, *then* look. That ordering is the spine of the whole
pipeline.

⚠️ **A warning you must internalise.** A t-SNE or UMAP plot is **for looking, not for measuring.**
The algorithm preserves who-is-near-whom locally, not global geometry. So:

- the **distance** between two blobs is not a real distance, do not read "these clusters are twice as
  far apart",
- the **size** of a blob is not a real density, t-SNE inflates sparse groups and shrinks dense ones,
- the **axes** have no units and no meaning.

Use it to spot groups and check that biology separates. For any quantitative claim, go back to the
numbers: the program activities, the differential test, the cluster compositions. We will hammer this
home at the end.

🔬 **TASK.** Run UMAP on the NMF embedding through **scanpy**, the exact tool the real pipeline uses.
UMAP on all 174k pixels would take minutes, so we work on a reproducible random subsample of 25,000
pixels, plenty to reveal the structure. We build the neighbour graph on `X_NMF`, then lay it out.

In [ ]:
# 🔬 your code here


Now colour the UMAP three ways on the same coordinates: by **section** (is there a batch split?),
by **condition**, and by the precomputed **lipizone** label (does biology form coherent blobs?). The
lipizone colouring is the truth we hope to see; the section colouring is the nuisance we will fix with
Harmony next.

In [ ]:
um = sub.obsm["X_umap"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))

# by section: each section is one MALDI acquisition (one batch)
for s, c in zip(sorted(sub.obs["SectionID"].unique()), ["tab:orange", "tab:purple"]):
    m = (sub.obs["SectionID"] == s).to_numpy()
    axes[0].scatter(um[m, 0], um[m, 1], s=3, alpha=0.5, color=c, label=f"section {s:.0f}", rasterized=True)
axes[0].set_title("coloured by section (the batch)"); axes[0].legend(markerscale=3)

# by condition
for cond, c in zip(["naive", "pregnant"], ["tab:blue", "tab:green"]):
    m = (sub.obs["Condition"] == cond).to_numpy()
    axes[1].scatter(um[m, 0], um[m, 1], s=3, alpha=0.5, color=c, label=cond, rasterized=True)
axes[1].set_title("coloured by condition"); axes[1].legend(markerscale=3)

# by precomputed lipizone colour: the biological structure we hope to see
axes[2].scatter(um[:, 0], um[:, 1], s=3, c=sub.obs["lipizone_color"], alpha=0.6, rasterized=True)
axes[2].set_title("coloured by lipizone (biology)")

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** The right panel shows many coloured islands: lipizones form coherent blobs, so
the embedding genuinely separates lipid territories. But look at the left panel: the two sections do
**not** fully overlap. Many blobs split into an orange part and a purple part side by side. That split
is the **batch effect**: the same biological territory landing in two places just because it was
measured on two different acquisitions. The middle panel shows the same split by condition, because in
our two-section design condition and section are one and the same.

❓ **QUESTION.** If we clustered on this embedding right now, some clusters would form around "which
section a pixel came from" instead of "what kind of tissue it is". Why would that wreck a later
control-vs-pregnant comparison, and which panel above is the evidence that we have a problem to fix?

### the same picture through t-SNE, the other flattener

We named two flatteners in the header, so we should run both. UMAP gave us the layout above; now we run
**t-SNE** on the *same* 25,000-pixel subsample and the *same* `X_NMF` embedding, so the only thing that
changes is the algorithm. t-SNE asks the same question UMAP does, "who are each pixel's near neighbours
in the 12-program space?", but answers it with a different optimisation: it turns neighbour relations
into probabilities and then arranges the 2-D points so those probabilities match. The two methods are
cousins, not twins.

🔬 **TASK.** Run `sc.tl.tsne` on `X_NMF` and scatter the result coloured by lipizone, next to the UMAP
we already made. The point is not that the two pictures look identical, they will not. The point is
that **both** flatteners pull the same biological islands apart, which tells us the structure lives in
the embedding, not in the quirks of one layout algorithm.

In [ ]:
# t-SNE on the SAME subsample and SAME embedding as the UMAP above (compress first, then look)
# 🔬 your code here


⚠️ **CHECKPOINT.** Two scatters, t-SNE on the left and UMAP on the right, both coloured by lipizone.
The shapes differ: t-SNE tends to scatter the blobs into many tight, well-separated puffs, UMAP knits
them into more connected continents. The exact positions are not the same and that is fine. What
matters is that the *same* coloured islands appear in both, so the biological separation is a property
of the embedding, not an artefact of one layout. This is also a live reminder of the warning above: if
two honest flatteners disagree this much on distances and blob sizes, you must never read a quantity
off either one. They are for looking, the numbers come from the embedding.

---
## 4 · Harmony: make the sections overlap, without erasing biology

**The problem.** Two sections, two acquisitions, two batches. Beyond real biology, each batch carries a
technical offset: a different day, a slightly different instrument state, tiny differences in matrix
deposition. In the embedding that offset shows up as the split you just saw, the same biological group
sitting in two side-by-side sub-clouds. We want pixels of the same biological type to mix across
sections, while keeping genuinely different territories apart.

**The idea.** Harmony works directly on the **embedding coordinates** (our `X_NMF`), never on the raw
lipids. It iterates two moves: (1) softly cluster all pixels together, ignoring which batch they came
from, then (2) within each soft cluster, nudge each batch's pixels toward the cluster's shared centre,
removing the part of the offset that is purely "which section". Repeat, and biologically identical
groups melt together while truly different groups stay separate.

🚨 **COURSE-CRITICAL DISTINCTION. Read this twice.**

Harmony's batch-corrected embedding is used **only for clustering and label transfer**. It is the space
in which we will find lipizones and carry labels from control to pregnant, because there we *want* the
two sections to coembed.

It is **never** used for the differential test. When we ask "which lipids changed in pregnancy?" we go
back to the **uMAIA-normalized, non-Harmonized** values in `adata.X`. The reason is sharp: with only
two sections, condition and batch are perfectly confounded, so Harmony cannot tell a real
pregnancy-versus-control difference from a technical section offset. Hand it the condition difference
and it will happily erase it as if it were noise. So Harmony cleans the space we *cluster* in, and the
differential test runs on the untouched data. The atlas states this explicitly: all differential
testing was done on the uMAIA output, with dimensionality reduction and harmonisation used solely for
clustering. Keep these two paths separate and you will never fool yourself.

🔬 **TASK.** Run Harmony on the NMF embedding, using **section** as the batch variable. The helper
`cl.embedding.harmonize` wraps `harmonypy` and returns the corrected embedding in the same
(pixels × programs) shape.

In [ ]:
# Harmony on the NMF embedding, batch = section. CLUSTERING/TRANSFER ONLY, never for differentials.
# 🔬 your code here


Now the before-and-after that proves it worked. We lay out the **Harmonized** embedding with UMAP
on the same 25,000 pixels and colour by section again. If Harmony did its job, the two sections should
now overlap where before they split, while the lipizone islands should stay distinct.

In [ ]:
# UMAP on the Harmonized embedding (same subsample), to compare against the pre-Harmony picture
sub.obsm["X_harmony"] = Z_harm[sub_idx].astype("float32")
sc.pp.neighbors(sub, n_neighbors=15, use_rep="X_harmony", random_state=SEED, key_added="harm")
sc.tl.umap(sub, neighbors_key="harm", random_state=SEED)
um_h = sub.obsm["X_umap"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for s, c in zip(sorted(sub.obs["SectionID"].unique()), ["tab:orange", "tab:purple"]):
    m = (sub.obs["SectionID"] == s).to_numpy()
    axes[0].scatter(um_h[m, 0], um_h[m, 1], s=3, alpha=0.5, color=c, label=f"section {s:.0f}", rasterized=True)
axes[0].set_title("AFTER Harmony: by section (now mixed)"); axes[0].legend(markerscale=3)

axes[1].scatter(um_h[:, 0], um_h[:, 1], s=3, c=sub.obs["lipizone_color"], alpha=0.6, rasterized=True)
axes[1].set_title("AFTER Harmony: by lipizone (biology kept)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout(); plt.show()

⚠️ **CHECKPOINT.** Left: the orange and purple pixels now interleave inside the blobs instead of
sitting in separate sub-clouds, the section offset has been absorbed. Right: the lipizone islands are
still distinct, the biology survived. That is exactly what "make the sections overlap without erasing
biology" means, and exactly why we cluster on this Harmonized space but test on the non-Harmonized
data.

To make the mixing quantitative rather than eyeballed, we measure, per UMAP layout, how
batch-segregated the local neighbourhoods are. For each pixel we look at its 15 nearest neighbours in
the layout and compute the fraction that come from the *other* section. If batches are perfectly
mixed, that fraction sits near 0.5 (a neighbour is a coin-flip between sections); if batches are
segregated, it sits near 0. Harmony should push the median fraction up toward 0.5.

In [ ]:
from sklearn.neighbors import NearestNeighbors

def cross_batch_fraction(coords2d, batch):
    # for each point, fraction of its 15 NN that belong to the other batch (-> 0.5 means well mixed)
    nn = NearestNeighbors(n_neighbors=16).fit(coords2d)
    _, ind = nn.kneighbors(coords2d)
    ind = ind[:, 1:]                                  # drop self
    same = (batch[ind] == batch[:, None]).mean(axis=1)
    return 1.0 - same                                 # fraction of neighbours from the OTHER batch

batch = sub.obs["SectionID"].to_numpy()
mix_before = cross_batch_fraction(um, batch)          # the pre-Harmony UMAP from section 3
mix_after = cross_batch_fraction(um_h, batch)         # the post-Harmony UMAP

print("                              BEFORE -> AFTER Harmony")
print(f"mean   cross-section neighbour fraction: {mix_before.mean():.3f} -> {mix_after.mean():.3f}")
print(f"median cross-section neighbour fraction: {np.median(mix_before):.3f} -> {np.median(mix_after):.3f}")
print("(0.5 = sections perfectly mixed locally; 0 = a pixel's neighbours are all its own section)")

⚠️ **CHECKPOINT.** Both numbers climb toward 0.5 after Harmony. Before, the median sits at 0:
more than half the pixels had *every* neighbour from their own section, the blobs were section-pure.
After Harmony the mean and median both rise sharply, so a typical pixel now sits next to neighbours
from *both* sections. That is the batch effect being removed, measured in a number rather than guessed
from a plot.

❓ **QUESTION.** Suppose we had instead handed Harmony `Condition` as the batch variable and it had
pushed this fraction all the way to 0.5. Why would that be a disaster for the pregnancy comparison, and
how does it connect to the rule that the differential test must run on `adata.X`, not on anything
Harmony touched?

---
## what we built, and where it goes next

We took two coronal sections, 174,768 pixels by 173 lipids, and walked the embedding pipeline end to
end, looking at the data at every step:

- **Feature selection by Moran's I.** We built spatial autocorrelation from a kNN graph and four lines
  of arithmetic, proved with side-by-side maps that high Moran means real anatomy and low Moran means
  noise, and kept the lipids above 0.4. Helper: `cl.analysis.morans_i`.
- **NMF.** We compressed the clean panel into 12 non-negative, additive lipid programs, showed on a toy
  matrix that NMF's recipes stay all-positive while PCA's components carry minus signs, read each
  program in two ways (its lipid recipe $H$ and its spatial activity map $W$), and applied the
  control-learned factors to both sections so they share one vocabulary. Helpers:
  `cl.embedding.seeded_nmf`, `cl.embedding.apply_nmf`.
- **t-SNE and UMAP.** We laid the embedding flat with both flatteners on the same subsample and saw the
  same lipizone islands in each, with the standing warning that distances, blob sizes, and axes carry
  no quantitative meaning. The layouts also exposed a section split.
- **Harmony.** We removed that section offset in the embedding space, kept the lipid territories
  distinct, and measured the mixing. We stated, twice, that Harmony is for clustering and label
  transfer only, never for the differential test, which always runs on the untouched `adata.X`. Helper:
  `cl.embedding.harmonize`.

The Harmonized embedding `adata_sel.obsm["X_harmony"]` is the input to the next notebook, where we
cluster it into **lipizones** and transfer those labels from control to pregnant. The differential
test that follows will reach back past Harmony to the raw `adata.X`. Two paths, kept clean, all the way
through.